In [12]:
# -*- coding: utf-8 -*-
"""
CSM triplet selection (min-dmid heuristic) + feature switch (angle/range) → (x,y)
+ Per-model runners (MLP, RidgeCV[linear/poly], RandomForest, GBR, GPR, QuantileGBR)
+ Save all test plots (incl. cc2-only) and logs to 'ML results/'

Notes:
- Triplet selection uses a deterministic geometric heuristic:
    * Among all candidates per cluster, choose the one whose midpoint
      is closest to the DBSCAN cluster centroid (min dmid).
- Toggle polynomial Ridge via RIDGE_USE_POLY / RIDGE_POLY_DEGREE.
- Saves metrics, predictions, and high-DPI plots for every model and both feature modes.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

# Sessions
TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2",'c5','c6','c7','c8','r1','r2','r3','r4']
TEST_SUFFIXES  = ["cc2"]  # explicit cc2 test plots generated per-model

# Radar roles
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]  # optional GT-only (not needed for min-dmid, but kept for completeness)

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# File column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.35
SPREAD_MAX_M     = 0.30
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

# Training knobs (MLP and others)
SEED        = 1337
VAL_SPLIT   = 0.15
BATCH_SIZE  = 64
LR_LOCAL    = 1e-3
WD          = 5e-9
EPOCHS_LOCAL  = 1000
PATIENCE_LOCAL= 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Ridge toggle
RIDGE_USE_POLY = True          # False = linear Ridge, True = PolynomialFeatures + Ridge
RIDGE_POLY_DEGREE = 2          # degree for polynomial Ridge

# GPR downsampling (for speed/memory)
GPR_MAX_TRAIN = 4000

# Output
OUT_DIR = "ML results"  # (user requested this exact folder name)

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta): 
    c,s=math.cos(theta), math.sin(theta); 
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

# =============== Binning + Clustering ===============

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

# =============== Candidate triplets per cluster ===============

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty: 
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok: 
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M: 
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

# =============== Triplet features (for dmid) ===============

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    We use index 4 (dmid) to implement min-dmid selector.
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# =============== Selection using min-dmid heuristic ===============

def select_triplets_for_suffix(sfx: str)->pd.DataFrame:
    """
    Deterministic triplet selection for a suffix:
    - Time-bin + DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (K-nearest per radar, time/spread gates).
        * For each candidate, compute features z.
        * Select candidate with minimal dmid (midpoint-to-cluster-centroid distance).
    """
    df = load_transform_suffix_all_radars(sfx)
    if df.empty: return pd.DataFrame()
    df = bin_time(df, BIN_SECONDS)
    rows=[]
    for tbin, d_bin in df.groupby("tbin"):
        dc = cluster_bin(d_bin)
        if dc.empty:
            continue
        cents = compute_centroids(dc)
        if not cents: continue
        for cid, d_cluster in dc.groupby("cluster"):
            if cid==-1: continue
            centroid = cents[cid]
            cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
            if len(cands)==0: 
                continue

            # --- min-dmid selection ---
            dmid_values = []
            for c in cands:
                z = triplet_features_for_scoring(c, centroid)
                dmid_values.append(z[4])  # index 4 = dmid
            best_idx = int(np.argmin(dmid_values))
            best = cands[best_idx]
            # ----------------------------

            # rebuild per_radar table to pick actual rows by index
            per_radar={}
            for r in FUSION_RADARS:
                dr = d_cluster[d_cluster["radar"]==r]
                if dr.empty:
                    per_radar = {}
                    break
                per_radar[r]=nearest_k_to_centroid(dr, centroid, K_NEAREST_PER_RADAR).reset_index(drop=True)
            if len(per_radar) < 3:
                continue

            idx1, idx2, idx3 = best["rows"]
            sel = [per_radar[FUSION_RADARS[0]].iloc[idx1],
                   per_radar[FUSION_RADARS[1]].iloc[idx2],
                   per_radar[FUSION_RADARS[2]].iloc[idx3]]

            points = best["points"]; times = best["times"]
            trip = {"suffix": sfx, "tbin": int(tbin), "cluster": int(cid),
                    "x_avg": float(points[:,0].mean()), "y_avg": float(points[:,1].mean()),
                    "t_ref": float(np.mean(times))}
            for r, srow in zip(FUSION_RADARS, sel):
                for k in ["angle","range","inten","snr","noise"]:
                    trip[f"{r}_{k}"]=float(srow[k])
            rows.append(trip)
    return pd.DataFrame(rows)

# =============== Localization models ===============

def build_Xy_from_triplets(df_sel: pd.DataFrame, feature_mode:str)->Tuple[np.ndarray,np.ndarray,List[str]]:
    if df_sel.empty: return np.empty((0,12)), np.empty((0,2)), []
    feat_cols=[]
    for r in FUSION_RADARS:
        if feature_mode=="angle":
            feat_cols += [f"{r}_angle", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        elif feature_mode=="range":
            feat_cols += [f"{r}_range", f"{r}_inten", f"{r}_snr", f"{r}_noise"]
        else:
            raise ValueError("FEATURE_MODE must be 'angle' or 'range'")
    X = df_sel[feat_cols].to_numpy(float)
    y = df_sel[["x_avg","y_avg"]].to_numpy(float)
    return X,y,feat_cols

class RangeToXY(nn.Module):
    def __init__(self, in_dim:int):
        super().__init__()
        self.fc1=nn.Linear(in_dim,128); self.ln1=nn.LayerNorm(128)
        self.fc2=nn.Linear(128,256);    self.ln2=nn.LayerNorm(256)
        self.fc3=nn.Linear(256,256);    self.ln3=nn.LayerNorm(256)
        self.fc4=nn.Linear(256,128);    self.ln4=nn.LayerNorm(128)
        self.fc5=nn.Linear(128,64);     self.ln5=nn.LayerNorm(64)
        self.out=nn.Linear(64,2)
        self.act=nn.LeakyReLU(0.1)
    def forward(self,x):
        x=self.act(self.ln1(self.fc1(x)))
        x=self.act(self.ln2(self.fc2(x)))
        res=x
        x=self.act(self.ln3(self.fc3(x)))
        x=x+res
        x=self.act(self.ln4(self.fc4(x)))
        x=self.act(self.ln5(self.fc5(x)))
        return self.out(x)

def train_mlp(Xtr_raw, ytr, Xval_raw, yval, tag, out_subdir):
    # preprocess (train-only fit)
    imputer = SimpleImputer(strategy="mean")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(imputer.fit_transform(Xtr_raw))
    Xval = scaler.transform(imputer.transform(Xval_raw))

    model = RangeToXY(in_dim=Xtr.shape[1]).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR_LOCAL, weight_decay=WD)
    crit  = nn.MSELoss()

    trX = torch.from_numpy(Xtr).float().to(DEVICE)
    trY = torch.from_numpy(ytr).float().to(DEVICE)
    vaX = torch.from_numpy(Xval).float().to(DEVICE)

    loader = DataLoader(TensorDataset(trX, trY), batch_size=BATCH_SIZE, shuffle=True)

    best = float("inf"); best_state=None; patience=PATIENCE_LOCAL
    hist = {"epoch": [], "train_mse": [], "val_mse": [], "val_smoothl1": []}

    def smooth_l1_np(y_true, y_pred, beta=1.0):
        d = np.abs(y_pred - y_true)
        h = np.where(d < beta, 0.5 * (d ** 2) / beta, d - 0.5 * beta)
        return float(h.mean())

    for ep in range(1, EPOCHS_LOCAL+1):
        model.train(); losses=[]
        for xb,yb in loader:
            opt.zero_grad(); pred=model(xb); loss=crit(pred,yb); loss.backward(); opt.step()
            losses.append(float(loss.item()))
        tr_mse = float(np.mean(losses))
        model.eval()
        with torch.no_grad():
            yp = model(vaX).detach().cpu().numpy()
        val_mse = float(np.mean((yp - yval)**2))
        val_s   = smooth_l1_np(yval, yp)

        hist["epoch"].append(ep); hist["train_mse"].append(tr_mse)
        hist["val_mse"].append(val_mse); hist["val_smoothl1"].append(val_s)

        if val_s < best:
            best=val_s; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience=PATIENCE_LOCAL
        else:
            patience-=1
        if patience<=0:
            print(f"[MLP] early stop at {ep}"); break

    if best_state: model.load_state_dict(best_state)

    # save training curves CSV + plots
    df_hist = pd.DataFrame(hist)
    ensure_dir(out_subdir)
    df_hist.to_csv(os.path.join(out_subdir, f"{tag}_MLP_training_curves.csv"), index=False)

    plt.figure(figsize=(7,3.6)); plt.plot(df_hist["epoch"], df_hist["train_mse"], label="train MSE")
    plt.plot(df_hist["epoch"], df_hist["val_mse"], label="val MSE"); plt.legend()
    plt.title(f"{tag} — MLP train/val MSE"); plt.tight_layout()
    savefig(os.path.join(out_subdir, f"{tag}_MLP_train_val_mse.png"))

    plt.figure(figsize=(7,3.2)); plt.plot(df_hist["epoch"], df_hist["val_smoothl1"], label="val SmoothL1")
    plt.legend(); plt.title(f"{tag} — MLP val SmoothL1"); plt.tight_layout()
    savefig(os.path.join(out_subdir, f"{tag}_MLP_val_smoothl1.png"))

    return model, {"imputer":imputer,"scaler":scaler, "Xval":Xval, "yval":yval}

# =============== Eval & plots ===============

def metrics_from_df(df: pd.DataFrame)->dict:
    dx = (df["x_pred"]-df["x_true"]).to_numpy(float)
    dy = (df["y_pred"]-df["y_true"]).to_numpy(float)
    err = np.hypot(dx,dy)
    return {
        "x_rmse": float(np.sqrt(np.mean(dx**2))),
        "y_rmse": float(np.sqrt(np.mean(dy**2))),
        "xy_rmse": float(np.sqrt(np.mean(err**2))),
        "xy_mae": float(np.mean(err)),
        "p90": float(np.percentile(err,90)),
        "N": int(len(df))
    }

def scatter_plot(y_true, y_pred, title, path):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true[:,0], y_true[:,1], s=10, label="true")
    plt.scatter(y_pred[:,0], y_pred[:,1], s=10, label="pred")
    plt.axis("equal"); plt.title(title); plt.legend(); plt.tight_layout()
    savefig(path)

def hist_plot(y_true, y_pred, title, path):
    err=np.hypot(y_pred[:,0]-y_true[:,0], y_pred[:,1]-y_true[:,1])
    plt.figure(figsize=(6,4))
    plt.hist(err, bins=40, alpha=0.9)
    plt.xlabel("||error|| [m]"); plt.title(title); plt.tight_layout()
    savefig(path)

def eval_model_on_suffixes(predict_fn, X_builder, suffixes, feature_mode, tag, out_subdir):
    all_rows=[]
    for sfx in suffixes:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[test {sfx}] no selected triplets")
            continue
        X_raw, y, feat_cols = X_builder(sel, feature_mode)
        Xs = predict_fn["prep"](X_raw)  # returns preprocessed X
        y_pred = predict_fn["pred"](Xs) # returns np array [N,2]

        df = pd.DataFrame({
            "suffix": sfx,
            "x_true": y[:,0], "y_true": y[:,1],
            "x_pred": y_pred[:,0], "y_pred": y_pred[:,1],
            "tbin": sel["tbin"].values, "cluster": sel["cluster"].values
        })
        all_rows.append(df)

        # save predictions CSV per suffix
        df.to_csv(os.path.join(out_subdir, f"{tag}_{feature_mode}_{predict_fn['name']}_{sfx}_predictions.csv"), index=False)

        # plots: overall test per suffix (scatter + hist)
        ytrue = y
        ypred = y_pred
        scatter_plot(ytrue, ypred,
                     f"Test {sfx} — {predict_fn['name']} ({feature_mode})",
                     os.path.join(out_subdir, f"{tag}_{feature_mode}_{predict_fn['name']}_{sfx}_test_scatter.png"))
        hist_plot(ytrue, ypred,
                  f"Test {sfx} — {predict_fn['name']} error ({feature_mode})",
                  os.path.join(out_subdir, f"{tag}_{feature_mode}_{predict_fn['name']}_{sfx}_test_errhist.png"))

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

# =============== High-level runner for one feature_mode ===============

def run_all_models_for_feature_mode(train_sel: pd.DataFrame, feature_mode: str, tag_base: str):
    """
    Trains/evals: MLP, RidgeCV(±poly), RandomForest, GBR, QuantileGBR, GPR.
    Saves:
      - test predictions & plots (incl. cc2-only)
      - metrics CSV (test)
      - train/val metrics CSV
      - train/val RMSE & MAE bar plots (per feature_mode)
      - MLP training curves
    """
    ensure_dir(OUT_DIR)
    out_subdir = os.path.join(OUT_DIR, f"{tag_base}_{feature_mode}")
    ensure_dir(out_subdir)

    # === Build X,y
    X_raw, y, feat_cols = build_Xy_from_triplets(train_sel, feature_mode)
    if len(X_raw)==0:
        raise RuntimeError(f"No training rows for feature_mode={feature_mode}")

    # shared train/val split for ALL models
    Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, random_state=SEED)

    # === 1) MLP (PyTorch)
    mlp_model, mlp_pp = train_mlp(Xtr_raw, ytr, Xval_raw, yval,
                                  tag=f"{tag_base}_{feature_mode}", out_subdir=out_subdir)
    mlp_predictor = {
        "name":"MLP",
        "prep": lambda X: mlp_pp["scaler"].transform(mlp_pp["imputer"].transform(X)),
        "pred": lambda Xs: mlp_model(torch.from_numpy(Xs).float().to(DEVICE)).detach().cpu().numpy()
    }

    # === 2) RidgeCV (linear or polynomial — toggled)
    imp = SimpleImputer(strategy="mean")
    sca = StandardScaler()
    Xtr0 = imp.fit_transform(Xtr_raw)
    Xval0 = imp.transform(Xval_raw)
    Xtr1 = sca.fit_transform(Xtr0)
    Xval1 = sca.transform(Xval0)

    if RIDGE_USE_POLY:
        poly = PolynomialFeatures(degree=RIDGE_POLY_DEGREE, include_bias=False)
        Xtr = poly.fit_transform(Xtr1)
        Xval = poly.transform(Xval1)
        ridge_name = f"RidgeCV_poly{RIDGE_POLY_DEGREE}"
    else:
        poly = None
        Xtr = Xtr1
        Xval = Xval1
        ridge_name = "RidgeCV"

    ridge = RidgeCV(alphas=np.logspace(-4, 3, 10), cv=5)
    ridge.fit(Xtr, ytr)

    def ridge_prep(X):
        X1 = sca.transform(imp.transform(X))
        return poly.transform(X1) if poly is not None else X1

    ridge_predictor = {
        "name": ridge_name,
        "prep": ridge_prep,
        "pred": lambda Xs: ridge.predict(Xs)
    }

    # === 3) RandomForest
    rf = RandomForestRegressor(n_estimators=300, max_depth=None, n_jobs=-1, random_state=SEED)
    imp_rf = SimpleImputer(strategy="mean")
    Xtr_rf = imp_rf.fit_transform(Xtr_raw)
    rf.fit(Xtr_rf, ytr)
    rf_predictor = {
        "name":"RandomForest",
        "prep": lambda X: imp_rf.transform(X),
        "pred": lambda Xs: rf.predict(Xs)
    }

    # === 4) GradientBoosting (squared error) with multi-output wrapper
    gbr_base = GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.1, max_depth=3, loss="squared_error"
    )
    gbr = MultiOutputRegressor(gbr_base)
    imp_gb = SimpleImputer(strategy="mean")
    Xtr_gb = imp_gb.fit_transform(Xtr_raw)
    Xval_gb = imp_gb.transform(Xval_raw)
    gbr.fit(Xtr_gb, ytr)
    gb_predictor = {
        "name":"GBR",
        "prep": lambda X: imp_gb.transform(X),
        "pred": lambda Xs: gbr.predict(Xs)
    }

    # === 5) Quantile GradientBoostingRegressor (median, loss="quantile")
    gbr_q_base = GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.1, max_depth=3,
        loss="quantile", alpha=0.5
    )
    gbr_q = MultiOutputRegressor(gbr_q_base)
    imp_gbq = SimpleImputer(strategy="mean")
    Xtr_gbq = imp_gbq.fit_transform(Xtr_raw)
    gbr_q.fit(Xtr_gbq, ytr)
    gbq_predictor = {
        "name":"GBR_Quantile50",
        "prep": lambda X: imp_gbq.transform(X),
        "pred": lambda Xs: gbr_q.predict(Xs)
    }

    # === 6) GaussianProcessRegressor (multi-output)
    imp_gpr = SimpleImputer(strategy="mean")
    sca_gpr = StandardScaler()
    Xtr_gpr_raw = imp_gpr.fit_transform(Xtr_raw)
    Xtr_gpr = sca_gpr.fit_transform(Xtr_gpr_raw)

    # optional downsample for GPR
    if len(Xtr_gpr) > GPR_MAX_TRAIN:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(Xtr_gpr), size=GPR_MAX_TRAIN, replace=False)
        Xtr_gpr_sub = Xtr_gpr[idx]
        ytr_gpr_sub = ytr[idx]
    else:
        Xtr_gpr_sub = Xtr_gpr
        ytr_gpr_sub = ytr

    kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=10.0, length_scale_bounds=(1e-2, 1e3)) \
             + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-5, 1e1))
    gpr_base = GaussianProcessRegressor(
        kernel=kernel, n_restarts_optimizer=2, alpha=1e-6,
        normalize_y=True, random_state=SEED
    )
    gpr = MultiOutputRegressor(gpr_base)
    gpr.fit(Xtr_gpr_sub, ytr_gpr_sub)

    def gpr_prep(X):
        Xg = imp_gpr.transform(X)
        return sca_gpr.transform(Xg)

    gpr_predictor = {
        "name":"GPR",
        "prep": gpr_prep,
        "pred": lambda Xs: gpr.predict(Xs)
    }

    # === Collect all predictors
    predictors = [mlp_predictor, ridge_predictor, rf_predictor, gb_predictor, gbq_predictor, gpr_predictor]

    # === Train/val metrics + bar plots
    tv_rows = []
    model_order = [p["name"] for p in predictors]
    for pred in predictors:
        name = pred["name"]
        Xtr_proc = pred["prep"](Xtr_raw)
        Xval_proc = pred["prep"](Xval_raw)
        ytr_pred = pred["pred"](Xtr_proc)
        yval_pred = pred["pred"](Xval_proc)

        df_tr = pd.DataFrame({
            "x_true": ytr[:,0], "y_true": ytr[:,1],
            "x_pred": ytr_pred[:,0], "y_pred": ytr_pred[:,1]
        })
        df_va = pd.DataFrame({
            "x_true": yval[:,0], "y_true": yval[:,1],
            "x_pred": yval_pred[:,0], "y_pred": yval_pred[:,1]
        })

        m_tr = metrics_from_df(df_tr)
        m_va = metrics_from_df(df_va)

        tv_rows.append({"model": name, "subset": "train", **m_tr})
        tv_rows.append({"model": name, "subset": "val",   **m_va})

    if tv_rows:
        df_tv = pd.DataFrame(tv_rows)
        df_tv.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_train_val_metrics.csv"), index=False)

        # bar plot: xy_rmse train vs val
        train_rmse = [df_tv[(df_tv["model"]==m) & (df_tv["subset"]=="train")]["xy_rmse"].values[0]
                      for m in model_order]
        val_rmse = [df_tv[(df_tv["model"]==m) & (df_tv["subset"]=="val")]["xy_rmse"].values[0]
                    for m in model_order]

        x = np.arange(len(model_order))
        width = 0.35
        plt.figure(figsize=(8,4))
        plt.bar(x - width/2, train_rmse, width, label="train")
        plt.bar(x + width/2, val_rmse,   width, label="val")
        plt.xticks(x, model_order, rotation=30, ha="right")
        plt.ylabel("XY RMSE [m]")
        plt.title(f"{tag_base} ({feature_mode}) — Train/Val XY RMSE")
        plt.legend()
        plt.tight_layout()
        savefig(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_train_val_xy_rmse.png"))

        # bar plot: xy_mae train vs val
        train_mae = [df_tv[(df_tv["model"]==m) & (df_tv["subset"]=="train")]["xy_mae"].values[0]
                     for m in model_order]
        val_mae = [df_tv[(df_tv["model"]==m) & (df_tv["subset"]=="val")]["xy_mae"].values[0]
                   for m in model_order]

        plt.figure(figsize=(8,4))
        plt.bar(x - width/2, train_mae, width, label="train")
        plt.bar(x + width/2, val_mae,   width, label="val")
        plt.xticks(x, model_order, rotation=30, ha="right")
        plt.ylabel("XY MAE [m]")
        plt.title(f"{tag_base} ({feature_mode}) — Train/Val XY MAE")
        plt.legend()
        plt.tight_layout()
        savefig(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_train_val_xy_mae.png"))

    # === Evaluate all on TEST suffixes (incl. cc2)
    metrics_rows=[]
    for pred in predictors:
        df_test = eval_model_on_suffixes(pred, build_Xy_from_triplets, TEST_SUFFIXES, feature_mode,
                                         tag=tag_base, out_subdir=out_subdir)
        if df_test.empty:
            continue
        # combined metrics
        m_all = metrics_from_df(df_test)
        metrics_rows.append({"feature_mode":feature_mode, "model":pred["name"], **m_all})
        # cc2-only metrics
        df_cc2 = df_test[df_test["suffix"]=="cc2"]
        if not df_cc2.empty:
            m_cc2 = metrics_from_df(df_cc2)
            metrics_rows.append({"feature_mode":feature_mode, "model":pred["name"]+"_cc2_only", **m_cc2})
        # save predictions
        df_test.to_csv(os.path.join(out_subdir,
                        f"{tag_base}_{feature_mode}_{pred['name']}_TEST_all_predictions.csv"), index=False)

    # === Save metrics CSV
    if metrics_rows:
        df_metrics = pd.DataFrame(metrics_rows)
        df_metrics.to_csv(os.path.join(out_subdir,
            f"{tag_base}_{feature_mode}_metrics.csv"), index=False)

# =============== Main ===============

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)
    print(f"[config] device={DEVICE}  ridge={'poly'+str(RIDGE_POLY_DEGREE) if RIDGE_USE_POLY else 'linear'}")
    print("[info] Triplet selector = min-dmid (geometric heuristic)")

    # 1) Build selected triplets for TRAIN using min-dmid
    sel_rows=[]
    for sfx in TRAIN_SUFFIXES:
        sel = select_triplets_for_suffix(sfx)
        if sel.empty:
            print(f"[train sel] {sfx}: 0 rows"); continue
        sel_rows.append(sel)
        print(f"[train sel] {sfx}: {len(sel)} rows")
    if not sel_rows:
        print("[abort] no selected triplets on TRAIN."); return
    train_sel = pd.concat(sel_rows, ignore_index=True)
    train_sel.to_csv(os.path.join(OUT_DIR, "train_selected_triplets_min_dmid.csv"), index=False)

    # 2) Run BOTH feature modes
    for FEATURE_MODE in ["angle","range"]:
        print(f"\n=== Running models for FEATURE_MODE={FEATURE_MODE} ===")
        run_all_models_for_feature_mode(train_sel, FEATURE_MODE, tag_base="CSM")

if __name__=="__main__":
    main()


[config] device=cuda  ridge=poly2
[info] Triplet selector = min-dmid (geometric heuristic)
[train sel] rr1: 46 rows
[train sel] rr2: 36 rows
[train sel] rr3: 53 rows
[train sel] rr4: 36 rows
[train sel] rr5: 39 rows
[train sel] rr6: 32 rows
[train sel] rr7: 31 rows
[train sel] rr8: 49 rows
[train sel] rr9: 55 rows
[train sel] rr12: 41 rows
[train sel] rr11: 38 rows
[train sel] cc3: 38 rows
[train sel] cc5: 53 rows
[train sel] cc6: 39 rows
[train sel] cc7: 36 rows
[train sel] cc8: 44 rows
[train sel] pcc3: 95 rows
[train sel] prr1: 39 rows
[train sel] prr2: 55 rows
[train sel] prr3: 100 rows
[train sel] prr4: 0 rows
[train sel] pcc1: 80 rows
[train sel] pcc2: 79 rows
[train sel] cc11: 56 rows
[train sel] cc12: 37 rows
[train sel] cc13: 39 rows
[train sel] cc10: 53 rows
[train sel] cc1: 50 rows
[train sel] cc2: 56 rows
[train sel] c5: 22 rows
[train sel] c6: 22 rows
[train sel] c7: 20 rows
[train sel] c8: 18 rows
[train sel] r1: 7 rows
[train sel] r2: 19 rows
[train sel] r3: 25 rows
[tra

In [9]:
# -*- coding: utf-8 -*-
"""
Triplet scorer benchmark
------------------------
Goal: Compare different triplet-selection strategies (scorers + heuristics) on a pure
triplet-ranking task, independent of the downstream (x,y) ML models.

Pipeline:
1) For each suffix in TRAIN_SUFFIXES:
    - Load + transform detections to world frame.
    - Bin in time; run DBSCAN per bin.
    - For each cluster:
        * Build candidate triplets (as in main script).
        * Compute "GT distance" for each candidate (to ref-centroid or fused-centroid).
        * Save group: (Z features, gt_dists, metadata).

2) Split groups into TRAIN / VAL.

3) Methods:
   - Baselines:
        * min_gt   (oracle: chooses candidate with lowest GT distance, used only as reference)
        * min_dmid (min midpoint-to-centroid distance)
        * min_spread
        * min_time (min internal time differences)
        * random
   - Learnable scorers:
        * TripletScorerMLP (listwise softmax)
        * TripletScorerLinear (listwise softmax)
        * GBR_scorer (GradientBoostingRegressor on -gt_dist)
        * RF_scorer  (RandomForestRegressor on -gt_dist)

4) Evaluation on VAL groups:
    - For each group and each method:
        * idx_best  = argmin(gt_dist)
        * idx_sel   = method's pick (argmax(score))
        * rank_sel  = rank of idx_sel when sorting gt_dist ascending
        * dist_best = gt_dist[idx_best]
        * dist_sel  = gt_dist[idx_sel]
        * gap       = dist_sel - dist_best
    - Aggregate metrics across groups:
        * top1_acc         = mean(idx_sel == idx_best)
        * mean_rank        = average rank_sel
        * mean_dist_best   = average dist_best
        * mean_dist_sel    = average dist_sel
        * mean_gap         = average gap

Outputs:
- Prints summary table.
- Saves CSV to OUT_DIR / "triplet_scorer_comparison.csv"

NOTE:
- This script re-implements some utilities from your main file so it's standalone.
  You can safely delete them and "from main_script import ..." if you prefer.
"""

import os, glob, math, random
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

import torch
import torch.nn as nn

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.linestyle": "--",
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ======================
# === Configuration  ===
# ======================

BASE_PATH = "/home/charithag/master thesis/practise/18Oct/corrected"

TRAIN_SUFFIXES = [
    "rr1","rr2","rr3","rr4","rr5","rr6","rr7","rr8","rr9","rr12","rr11",
    "cc3","cc5","cc6","cc7","cc8","pcc3","prr1","prr2","prr3","prr4",
    "pcc1","pcc2","cc11","cc12","cc13","cc10","cc1","cc2"
]

# Fusion + reference radars (same roles as in main script)
FUSION_RADARS   = ["rpi4","rpi1","rpi3"]
REFERENCE_RADARS= ["radar1","rpi2"]

# Anchors + yaws (deg CCW+)
radar_positions: Dict[str, Tuple[float, float]] = {
    "rpi4": (0.0, 0.0),
    "rpi1": (0.07, 0.7),
    "rpi3": (-0.04, -0.75),
    "rpi5": (2.3, 0.4),
    "rpi6": (2.8, 1.7),
}
sides = {r: {"angle": 0.0} for r in radar_positions.keys()}

# CSV column mapping (ti_mmwave-style, 0-based)
CSV_X_COL        = 3
CSV_Y_COL        = 4
CSV_RANGE_COL    = 6
CSV_DOPPLER_COL  = 8
CSV_ANGLE_COL    = 9          # (1-based col10)
CSV_INTEN_COL    = 10         # (1-based col11)
CSV_SNR_COL      = 11         # (1-based col12)
CSV_NOISE_COL    = 12         # (1-based col13)
CSV_TIME_COL     = -1

# Pipeline knobs (same as main, or tweak if you like)
BIN_SECONDS      = 0.20
DBSCAN_EPS       = 0.30
DBSCAN_MINPTS    = 3
TIME_TOLERANCE_S = 0.35
SPREAD_MAX_M     = 0.30
K_NEAREST_PER_RADAR = 3
REMOVE_DOPPLER_EQ: Optional[float] = 8.0

SEED        = 1337
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR     = "ML results"  # reuse same folder

# Training knobs for listwise torch scorers
EPOCHS_LISTWISE = 1000
LR_SCORER       = 1e-3
WD_SCORER       = 1e-8

# ======================
# === Utilities      ===
# ======================

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def savefig(path): ensure_dir(os.path.dirname(path)); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close()

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def _has_header(path: str) -> bool:
    try:
        with open(path,"r",errors="ignore") as f:
            toks = (f.readline().strip()).split(",")
        for t in toks:
            try: float(t)
            except ValueError: return True
        return False
    except: return False

def _read_any(path: str, header: Optional[int]) -> pd.DataFrame:
    try:    return pd.read_csv(path, header=header, sep=None, engine="python")
    except: return pd.read_csv(path, header=header, delim_whitespace=True)

def _get(df, spec):
    if isinstance(spec,int):
        idx = spec if spec>=0 else df.shape[1]+spec
        return df.iloc[:,idx]
    if isinstance(spec,str):
        if spec in df.columns: return df[spec]
        low = {c.lower():c for c in df.columns}
        if spec.lower() in low: return df[low[spec.lower()]]
    raise KeyError(spec)

def discover_files_for_radar(base: str, radar: str, suffix: str)->List[str]:
    exact = os.path.join(base, radar, suffix)
    if os.path.isfile(exact): return [exact]
    if os.path.isdir(exact):
        files = glob.glob(os.path.join(exact,"**","*"), recursive=True)
        return sorted([f for f in files if os.path.isfile(f)])
    root = os.path.join(base, radar)
    if os.path.isdir(root):
        candidates = glob.glob(os.path.join(root,"**","*"), recursive=True)
        hits = [f for f in candidates if os.path.isfile(f) and os.path.basename(f)==suffix]
        return sorted(hits)
    return []

def load_one_file(path: str) -> pd.DataFrame:
    header = 0 if _has_header(path) else None
    df = _read_any(path, header)
    out = pd.DataFrame({
        "x":        pd.to_numeric(_get(df, CSV_X_COL), errors="coerce"),
        "y":        pd.to_numeric(_get(df, CSV_Y_COL), errors="coerce"),
        "range":    pd.to_numeric(_get(df, CSV_RANGE_COL), errors="coerce"),
        "angle":    pd.to_numeric(_get(df, CSV_ANGLE_COL), errors="coerce"),
        "inten":    pd.to_numeric(_get(df, CSV_INTEN_COL), errors="coerce"),
        "snr":      pd.to_numeric(_get(df, CSV_SNR_COL), errors="coerce"),
        "noise":    pd.to_numeric(_get(df, CSV_NOISE_COL), errors="coerce"),
    })
    try: out["doppler"] = pd.to_numeric(_get(df, CSV_DOPPLER_COL), errors="coerce")
    except: out["doppler"] = np.nan
    try: out["timestamp"] = _get(df, CSV_TIME_COL)
    except: out["timestamp"] = np.nan
    return out.dropna(subset=["x","y"]).reset_index(drop=True)

def load_radar_suffix(base: str, radar: str, suffix: str) -> pd.DataFrame:
    files = discover_files_for_radar(base, radar, suffix)
    if not files:
        return pd.DataFrame(columns=["x","y","range","angle","inten","snr","noise","timestamp","radar"])
    parts=[]
    for f in files:
        try:
            df = load_one_file(f)
            if REMOVE_DOPPLER_EQ is not None and "doppler" in df.columns:
                df = df[~np.isclose(df["doppler"].astype(float), REMOVE_DOPPLER_EQ)]
            df["radar"]=radar
            parts.append(df)
        except Exception as e:
            print(f"[warn] load {f}: {e}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def rot2d(theta): 
    c,s=math.cos(theta), math.sin(theta); 
    return np.array([[c,-s],[s,c]],float)

def transform_local_to_world(df: pd.DataFrame, yaw_deg: float, tx: float, ty: float)->pd.DataFrame:
    R = rot2d(math.radians(yaw_deg))
    xy = df[["x","y"]].to_numpy(float) @ R.T
    out = df.copy()
    out["xw"]=xy[:,0]+tx; out["yw"]=xy[:,1]+ty
    return out

def load_transform_suffix_all_radars(sfx: str) -> pd.DataFrame:
    parts=[]
    for r,(tx,ty) in radar_positions.items():
        d = load_radar_suffix(BASE_PATH, r, sfx)
        if d.empty: continue
        yaw = float(sides.get(r,{}).get("angle",0.0))
        d = transform_local_to_world(d, yaw, tx, ty)
        parts.append(d)
    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    return df[["x","y","range","angle","inten","snr","noise","timestamp","radar","xw","yw"]]

def to_seconds(s: pd.Series)->pd.Series:
    a = s.astype(str).str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
    return pd.to_numeric(a, errors="coerce")

def bin_time(df: pd.DataFrame, bin_s: float)->pd.DataFrame:
    df = df.dropna(subset=["timestamp"]).copy()
    df["t"] = to_seconds(df["timestamp"])
    df = df.dropna(subset=["t"])
    df["tbin"] = np.floor(df["t"]/bin_s).astype(int)
    return df

def cluster_bin(df_bin: pd.DataFrame)->pd.DataFrame:
    if df_bin.empty: return pd.DataFrame()
    X = df_bin[["xw","yw"]].to_numpy(float)
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(X)
    out = df_bin.copy()
    out["cluster"]=labels
    return out

def compute_centroids(df_bin: pd.DataFrame)->Dict[int, Tuple[float,float]]:
    cents={}
    for c, d in df_bin.groupby("cluster"):
        if c==-1: continue
        cents[c]=(float(d["xw"].mean()), float(d["yw"].mean()))
    return cents

def nearest_k_to_centroid(d: pd.DataFrame, centroid: Tuple[float,float], k:int)->pd.DataFrame:
    dx = d["xw"].to_numpy()-centroid[0]
    dy = d["yw"].to_numpy()-centroid[1]
    dists = np.hypot(dx,dy)
    idx = np.argsort(dists)[:min(k,len(d))]
    return d.iloc[idx].copy()

def build_candidates_for_cluster(d_cluster: pd.DataFrame,
                                 centroid: Tuple[float,float],
                                 fusion_radars: List[str],
                                 k_each:int=K_NEAREST_PER_RADAR)->List[dict]:
    per_radar={}
    for r in fusion_radars:
        dr = d_cluster[d_cluster["radar"]==r]
        if dr.empty: 
            return []  # must have all fusion radars
        kn = nearest_k_to_centroid(dr, centroid, k_each).reset_index(drop=True)
        per_radar[r]=kn

    R1,R2,R3 = fusion_radars[:3]
    cands=[]
    n1, n2, n3 = len(per_radar[R1]), len(per_radar[R2]), len(per_radar[R3])
    for ia in range(n1):
        a = per_radar[R1].iloc[ia]
        for ib in range(n2):
            b = per_radar[R2].iloc[ib]
            for ic in range(n3):
                c = per_radar[R3].iloc[ic]
                # time gate
                t_ok = (abs(a["t"]-b["t"])<=TIME_TOLERANCE_S and
                        abs(a["t"]-c["t"])<=TIME_TOLERANCE_S)
                if not t_ok: 
                    continue
                # spread gate
                P = np.array([[a["xw"],a["yw"]],
                              [b["xw"],b["yw"]],
                              [c["xw"],c["yw"]]], float)
                d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
                spread = max(d01,d02,d12)
                if spread>SPREAD_MAX_M: 
                    continue
                cands.append({
                    "rows": (ia, ib, ic),
                    "points": P,                 # [3,2] world coords
                    "times": (float(a["t"]), float(b["t"]), float(c["t"])),
                    "centroid": (float(P[:,0].mean()), float(P[:,1].mean()))
                })
    return cands

def gt_centroids_from_reference(df_bin: pd.DataFrame)->List[Tuple[float,float]]:
    d = df_bin[df_bin["radar"].isin(REFERENCE_RADARS)]
    if d.empty: return []
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS).fit_predict(d[["xw","yw"]].to_numpy())
    cents=[]
    for c in np.unique(labels):
        if c==-1: continue
        dd = d[labels==c]
        cents.append((float(dd["xw"].mean()), float(dd["yw"].mean())))
    return cents

def triplet_features_for_scoring(cand: dict, cluster_centroid: Tuple[float,float])->np.ndarray:
    """
    z = [d01, d02, d12, spread, dmid, |t0-t1|, |t0-t2|, |t1-t2|]
    """
    P = cand["points"]; t0,t1,t2 = cand["times"]
    d01 = np.hypot(*(P[0]-P[1])); d02 = np.hypot(*(P[0]-P[2])); d12 = np.hypot(*(P[1]-P[2]))
    spread = max(d01,d02,d12)
    mid = P.mean(axis=0)
    dmid = np.hypot(*(mid - np.array(cluster_centroid)))
    dt = np.array([abs(t0-t1), abs(t0-t2), abs(t1-t2)], float)
    z = np.array([d01,d02,d12,spread,dmid, dt[0],dt[1],dt[2]], float)
    return z

# ======================
# === Triplet Groups ===
# ======================

def collect_triplet_groups(suffixes: List[str]):
    """
    Returns list of groups.
    Each group = {
        "Z":        np.ndarray [nC, in_dim],
        "gt_dists": np.ndarray [nC],
        "meta":     (suffix, tbin, cluster_id)
    }
    gt_dist for a candidate = distance between its centroid and:
        - nearest reference centroid (if available), else
        - DBSCAN cluster centroid (cents[cid])
    """
    groups=[]
    for sfx in suffixes:
        df = load_transform_suffix_all_radars(sfx)
        if df.empty:
            print(f"[groups] {sfx}: no data")
            continue
        df = bin_time(df, BIN_SECONDS)
        for tbin, d_bin in df.groupby("tbin"):
            dc = cluster_bin(d_bin)
            if dc.empty:
                continue
            cents = compute_centroids(dc)
            if not cents: 
                continue
            gt_cents = gt_centroids_from_reference(dc)
            for cid, d_cluster in dc.groupby("cluster"):
                if cid==-1: 
                    continue
                centroid = cents[cid]
                cands = build_candidates_for_cluster(d_cluster, centroid, FUSION_RADARS, K_NEAREST_PER_RADAR)
                if len(cands)==0:
                    continue
                Z  = []
                gd = []
                for cand in cands:
                    z = triplet_features_for_scoring(cand, centroid)
                    Z.append(z)
                    if gt_cents:
                        dmin = min(np.hypot(*(np.array(cand["centroid"])-np.array(g))) for g in gt_cents)
                    else:
                        dmin = np.hypot(*(np.array(cand["centroid"])-np.array(centroid)))
                    gd.append(dmin)
                Z = np.stack(Z, axis=0)
                gd = np.array(gd, float)
                groups.append({
                    "Z": Z,
                    "gt_dists": gd,
                    "meta": (sfx, int(tbin), int(cid))
                })
        print(f"[groups] {sfx}: collected {sum(1 for g in groups if g['meta'][0]==sfx)} groups")
    print(f"[groups] total groups = {len(groups)}")
    return groups

# ==============================
# === Listwise torch scorers ===
# ==============================

class TripletScorerMLP(nn.Module):
    def __init__(self, in_dim=8, hid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid), nn.ReLU(),
            nn.Linear(hid, hid), nn.ReLU(),
            nn.Linear(hid, 1)
        )
    def forward(self, z):  # [B, in_dim]
        return self.net(z).squeeze(-1)  # [B]

class TripletScorerLinear(nn.Module):
    def __init__(self, in_dim=8):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, z):
        return self.fc(z).squeeze(-1)

def train_listwise_scorer(model: nn.Module, groups_train, name: str):
    """
    Listwise softmax training, target = argmin(gt_dist)
    """
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR_SCORER, weight_decay=WD_SCORER)

    def group_loss(Z_np, gt_d):
        # target index = argmin(gt_dist)
        y_idx = int(np.argmin(gt_d))
        Zt = torch.from_numpy(Z_np).float().to(DEVICE)  # [nC, in_dim]
        s  = model(Zt)                                  # [nC]
        logp = s - torch.logsumexp(s, dim=0)
        return -logp[y_idx]

    losses=[]
    for ep in range(1, EPOCHS_LISTWISE+1):
        random.shuffle(groups_train)
        loss_sum = 0.0
        for g in groups_train:
            Z_np = g["Z"]; gd = g["gt_dists"]
            opt.zero_grad()
            loss = group_loss(Z_np, gd)
            loss.backward()
            opt.step()
            loss_sum += float(loss.item())
        loss_avg = loss_sum / max(len(groups_train),1)
        losses.append(loss_avg)
        if ep % 50 == 0 or ep == 1:
            print(f"[{name}] epoch {ep:3d}  loss={loss_avg:.4f}")

    return model, losses

# ===================================
# === Pointwise sklearn scorers   ===
# ===================================

def build_pointwise_dataset(groups):
    """
    Flatten (Z, gt_dist) across groups -> X, y
    Target y = -gt_dist (so larger is better).
    """
    X_list=[]; y_list=[]
    for g in groups:
        Z = g["Z"]
        gd = g["gt_dists"]
        X_list.append(Z)
        y_list.append(-gd)   # want max score = min distance
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y

# ==============================
# === Evaluation of methods  ===
# ==============================

def evaluate_methods(groups_val, methods):
    """
    methods: list of dicts
        {
          "name": str,
          "score_fn": callable(Z_np) -> np.ndarray [nC]
        }
    """
    records=[]
    for g in groups_val:
        Z = g["Z"]
        gd = g["gt_dists"]
        idx_best = int(np.argmin(gd))
        # sort gt_dists ascending to define rank
        order = np.argsort(gd)
        rank_of = {int(order[i]): i+1 for i in range(len(order))}  # 1-based

        for m in methods:
            s = m["score_fn"](Z)
            idx_sel = int(np.argmax(s))
            dist_best = float(gd[idx_best])
            dist_sel  = float(gd[idx_sel])
            rank_sel  = int(rank_of[idx_sel])
            top1 = 1 if idx_sel == idx_best else 0
            gap = dist_sel - dist_best
            records.append({
                "method": m["name"],
                "n_cands": len(gd),
                "top1": top1,
                "rank_sel": rank_sel,
                "dist_best": dist_best,
                "dist_sel": dist_sel,
                "gap": gap
            })
    df = pd.DataFrame(records)
    summary = df.groupby("method").agg(
        n_groups=("top1","count"),
        avg_cands=("n_cands","mean"),
        top1_acc=("top1","mean"),
        mean_rank=("rank_sel","mean"),
        mean_dist_best=("dist_best","mean"),
        mean_dist_sel=("dist_sel","mean"),
        mean_gap=("gap","mean"),
    ).reset_index()
    return df, summary

# ======================
# === Main benchmark ===
# ======================

def main():
    set_seed(SEED)
    ensure_dir(OUT_DIR)

    print("[*] Collecting triplet groups...")
    groups = collect_triplet_groups(TRAIN_SUFFIXES)
    if not groups:
        print("[abort] no groups collected."); return

    # train/val split on groups
    groups_train, groups_val = train_test_split(groups, test_size=0.3, random_state=SEED)
    print(f"[split] train_groups = {len(groups_train)}, val_groups = {len(groups_val)}")

    # === Baseline methods ===
    def baseline_min_dmid(Z, gd_unused, g):
        # recompute dmid per candidate using cand.points and cluster centroid
        # but we don't have cand here; so we instead rely on features:
        # features = [d01, d02, d12, spread, dmid, dt01, dt02, dt12]
        # index 4 = dmid
        dmid = Z[:,4]
        return -dmid  # smaller dmid is better -> higher score

    def baseline_min_spread(Z, gd_unused, g):
        spread = Z[:,3]  # index 3 = spread
        return -spread

    def baseline_min_time(Z, gd_unused, g):
        # use sum of dt's as time dispersion
        dt_sum = Z[:,5] + Z[:,6] + Z[:,7]
        return -dt_sum

    def baseline_random(Z, gd_unused, g):
        r = np.random.rand(len(Z))
        return r

    def baseline_oracle(Z, gd, g):
        # ideal method: score = -gt_dist
        return -gd

    # We'll wrap them so they match score_fn(Z) signature.
    # But some need gt_dists, so we adapt below in evaluation.

    # === Train listwise scorers ===
    print("[*] Training listwise MLP scorer...")
    mlp_model, mlp_losses = train_listwise_scorer(TripletScorerMLP(in_dim=8),
                                                  groups_train, name="MLP")
    print("[*] Training listwise Linear scorer...")
    lin_model, lin_losses = train_listwise_scorer(TripletScorerLinear(in_dim=8),
                                                  groups_train, name="Linear")

    # === Train pointwise tree scorers ===
    print("[*] Building pointwise dataset for tree scorers...")
    X_train, y_train = build_pointwise_dataset(groups_train)

    print("[*] Training GBR scorer...")
    gbr = GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.1, max_depth=3, loss="squared_error"
    )
    gbr.fit(X_train, y_train)

    print("[*] Training RF scorer...")
    rf = RandomForestRegressor(
        n_estimators=300, max_depth=None, n_jobs=-1, random_state=SEED
    )
    rf.fit(X_train, y_train)

    # === Define methods for evaluation ===
    methods = []

    # Oracle (theoretical upper bound)
    methods.append({
        "name": "oracle_min_gt",
        "score_fn": lambda Z, g=None: None  # will be replaced in eval loop
    })
    # No-training baselines
    methods.append({
        "name": "baseline_min_dmid",
        "score_fn": lambda Z, g=None: None
    })
    methods.append({
        "name": "baseline_min_spread",
        "score_fn": lambda Z, g=None: None
    })
    methods.append({
        "name": "baseline_min_time",
        "score_fn": lambda Z, g=None: None
    })
    methods.append({
        "name": "baseline_random",
        "score_fn": lambda Z, g=None: None
    })

    # Listwise torch scorers
    mlp_model.eval()
    lin_model.eval()
    methods.append({
        "name": "MLP_listwise",
        "score_fn": lambda Z, g=None: mlp_model(torch.from_numpy(Z).float().to(DEVICE)).detach().cpu().numpy()
    })
    methods.append({
        "name": "Linear_listwise",
        "score_fn": lambda Z, g=None: lin_model(torch.from_numpy(Z).float().to(DEVICE)).detach().cpu().numpy()
    })

    # Tree-based scorers (pointwise)
    methods.append({
        "name": "GBR_pointwise",
        "score_fn": lambda Z, g=None: gbr.predict(Z)
    })
    methods.append({
        "name": "RF_pointwise",
        "score_fn": lambda Z, g=None: rf.predict(Z)
    })

    # === Custom evaluation loop (so baselines can see gt_dists) ===
    records=[]
    for g in groups_val:
        Z = g["Z"]
        gd = g["gt_dists"]
        idx_best = int(np.argmin(gd))
        order = np.argsort(gd)
        rank_of = {int(order[i]): i+1 for i in range(len(order))}

        for m in methods:
            name = m["name"]
            if name == "oracle_min_gt":
                scores = -gd
            elif name == "baseline_min_dmid":
                scores = baseline_min_dmid(Z, gd, g)
            elif name == "baseline_min_spread":
                scores = baseline_min_spread(Z, gd, g)
            elif name == "baseline_min_time":
                scores = baseline_min_time(Z, gd, g)
            elif name == "baseline_random":
                scores = baseline_random(Z, gd, g)
            else:
                scores = m["score_fn"](Z, g)

            idx_sel = int(np.argmax(scores))
            dist_best = float(gd[idx_best])
            dist_sel  = float(gd[idx_sel])
            rank_sel  = int(rank_of[idx_sel])
            top1 = 1 if idx_sel == idx_best else 0
            gap = dist_sel - dist_best
            records.append({
                "method": name,
                "n_cands": len(gd),
                "top1": top1,
                "rank_sel": rank_sel,
                "dist_best": dist_best,
                "dist_sel": dist_sel,
                "gap": gap
            })

    df = pd.DataFrame(records)
    summary = df.groupby("method").agg(
        n_groups=("top1","count"),
        avg_cands=("n_cands","mean"),
        top1_acc=("top1","mean"),
        mean_rank=("rank_sel","mean"),
        mean_dist_best=("dist_best","mean"),
        mean_dist_sel=("dist_sel","mean"),
        mean_gap=("gap","mean"),
    ).reset_index()

    # Save + print
    ensure_dir(OUT_DIR)
    out_csv = os.path.join(OUT_DIR, "triplet_scorer_comparison.csv")
    summary.to_csv(out_csv, index=False)
    print("\n=== Triplet scorer comparison (VAL groups) ===")
    print(summary.to_string(index=False))
    print(f"\nSaved summary to: {out_csv}")

    # Optional: quick bar plot of top1_acc
    plt.figure(figsize=(8,4))
    x = np.arange(len(summary))
    plt.bar(x, summary["top1_acc"])
    plt.xticks(x, summary["method"], rotation=30, ha="right")
    plt.ylabel("Top-1 accuracy")
    plt.title("Triplet selection — methods comparison (VAL)")
    plt.tight_layout()
    savefig(os.path.join(OUT_DIR, "triplet_scorer_top1_accuracy.png"))

if __name__=="__main__":
    main()


[*] Collecting triplet groups...
[groups] rr1: collected 46 groups
[groups] rr2: collected 36 groups
[groups] rr3: collected 53 groups
[groups] rr4: collected 36 groups
[groups] rr5: collected 39 groups
[groups] rr6: collected 32 groups
[groups] rr7: collected 31 groups
[groups] rr8: collected 49 groups
[groups] rr9: collected 55 groups
[groups] rr12: collected 41 groups
[groups] rr11: collected 38 groups
[groups] cc3: collected 38 groups
[groups] cc5: collected 53 groups
[groups] cc6: collected 39 groups
[groups] cc7: collected 36 groups
[groups] cc8: collected 44 groups
[groups] pcc3: collected 95 groups
[groups] prr1: collected 39 groups
[groups] prr2: collected 55 groups
[groups] prr3: collected 100 groups
[groups] prr4: no data
[groups] pcc1: collected 80 groups
[groups] pcc2: collected 79 groups
[groups] cc11: collected 56 groups
[groups] cc12: collected 37 groups
[groups] cc13: collected 39 groups
[groups] cc10: collected 53 groups
[groups] cc1: collected 50 groups
[groups] cc2: